<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/1_bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Bit LLM

In [73]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

In [74]:
class BitLinear(nn.Linear):
  def forward(self, x: torch.Tensor):
    # Scale input tensor down to 8 bits
    i_scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_quant = torch.round(x * 127 / i_scale).clamp(-128, 127)

    # Scale layer weights to ternary -,+,0
    w = self.weight
    w_scale = w.abs().mean()
    w_quant = torch.round(w / (w_scale + 1e-5)).clamp(-1, 1)
    w_ternary = w + (w_quant - w).detach()

    out_t = F.linear(x_quant, w_ternary, self.bias)

    # Scale output tensor up to 8bits
    out_t = out_t * (i_scale / 127) * (w_scale)
    return out_t

Tests for the BitLinear class

In [75]:
bit_linear_layer = BitLinear(10, 5)
x = torch.randn(1, 10)
output = bit_linear_layer(x)

assert isinstance(output, torch.Tensor), "Assertion Failed: Output should be a torch.Tensor"
assert output.shape == (1, 5), f"Assertion Failed: Output shape mismatch. Expected (1, {5}), got {output.shape}"
assert not torch.isnan(output).any(), "Assertion Failed: Output contains NaN values"
assert not torch.isinf(output).any(), "Assertion Failed: Output contains Inf values"
print(f"Output tensor (first 5 elements): {output.flatten()[:5].tolist()}")

Output tensor (first 5 elements): [1.2482692003250122, -1.0667444467544556, 0.1194518432021141, -0.44655945897102356, -0.3752269744873047]


Replace Linear Layers with BitLinear

In [76]:
def replace_linears_with_bitlinear(model: nn.Module):
  for name, module in model.named_children():
    if isinstance(module, nn.Linear):
      has_bias = module.bias is not None
      bit_layer = BitLinear(module.in_features, module.out_features, bias=has_bias)
      setattr(model, name, bit_layer)
    else:
      replace_linears_with_bitlinear(module)

Test the replacement function

In [77]:
class TestModule(nn.Module):
  def __init__(self):
    super().__init__()
    self.l1, self.relu, self.l2 = nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 5)

  def forward(self, x):
    return self.l2(self.relu(self.l1(x)))

original_model = TestModule()

print("Original model structure:")
print(original_model)

replace_linears_with_bitlinear(original_model)

print("\nModel structure after replacement:")
print(original_model)

assert isinstance(original_model.l1, BitLinear), "l1 was not replaced by BitLinear"
assert isinstance(original_model.l2, BitLinear), "l2 was not replaced by BitLinear"

Original model structure:
TestModule(
  (l1): Linear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (l2): Linear(in_features=20, out_features=5, bias=True)
)

Model structure after replacement:
TestModule(
  (l1): BitLinear(in_features=10, out_features=20, bias=True)
  (relu): ReLU()
  (l2): BitLinear(in_features=20, out_features=5, bias=True)
)


Setitng up the Transformer Blocks

Tests for RMS Norm Layer

In [78]:
class RMSNorm(nn.Module):
  def __init__(self, dim: int, eps = 1e-6):
    super().__init__()
    self.eps = eps
    self.weight = nn.Parameter(torch.ones(dim))
  def forward(self, x: torch.Tensor):
    var = x.pow(2).mean(dim=-1, keepdim=True)
    return x * torch.rsqrt(var + self.eps) * self.weight

In [79]:
rms_norm_layer = RMSNorm(10)
x_rms = torch.randn(1, 10)
output_rms = rms_norm_layer(x_rms)
assert output_rms.shape == (1, 10), f"Assertion Failed: RMSNorm output shape mismatch. Expected (1, 10), got {output_rms.shape}"
assert not torch.isnan(output_rms).any(), "Assertion Failed: RMSNorm output contains NaN values"
assert not torch.isinf(output_rms).any(), "Assertion Failed: RMSNorm output contains Inf values"
print(f"RMSNorm Output tensor (first 5 elements): {output_rms.flatten()[:5].tolist()}")

RMSNorm Output tensor (first 5 elements): [1.1829748153686523, -1.4885714054107666, 0.8041207194328308, -0.14940185844898224, -0.008178023621439934]


BitNet Causal Self Attention

In [80]:
class CausalSelfAttention(nn.Module):
  def __init__(self, dim: int, n_heads: int):
    super().__init__()
    self.dim = dim
    self.n_heads = n_heads
    self.head_dim = dim // n_heads
    self.qkv_proj = BitLinear(dim, 3 * dim, bias=False)
    self.out_proj = BitLinear(dim, dim, bias=False)
  def forward(self, x):
    B, T, C = x.shape
    qkv = self.qkv_proj(x)
    q, k, v = qkv.split(self.dim, dim=-1)

    q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
    v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    mask = torch.tril(torch.ones(T, T), diagonal=0).bool()
    scores = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim)
    scores = scores.masked_fill(~mask, float("-inf"))
    probs = F.softmax(scores, dim=-1)
    context = probs @ v
    context = context.transpose(1, 2).contiguous().view(B, T, C)
    return self.out_proj(context)

Tests for the CausalSelfAttention class

In [81]:
attention_layer = CausalSelfAttention(dim=64, n_heads=4)
x_attn = torch.randn(1, 10, 64)
output_attn = attention_layer(x_attn)
assert isinstance(output_attn, torch.Tensor), "Assertion Failed: CausalSelfAttention output should be a torch.Tensor"
assert output_attn.shape == (1, 10, 64), f"Assertion Failed: CausalSelfAttention output shape mismatch. Expected (1, 10, 64), got {output_attn.shape}"
assert not torch.isnan(output_attn).any(), "Assertion Failed: CausalSelfAttention output contains NaN values"
assert not torch.isinf(output_attn).any(), "Assertion Failed: CausalSelfAttention output contains Inf values"
print(f"CausalSelfAttention Output tensor shape: {output_attn.shape}")

CausalSelfAttention Output tensor shape: torch.Size([1, 10, 64])


BitNet Feed Forward Layer

In [84]:
from prompt_toolkit.lexers import base
class BitFFN(nn.Module):
  def __init__(self, dim:int, h_dim: int):
    super().__init__()
    self.gate_proj = BitLinear(dim, h_dim, bias=False)
    self.up_proj = BitLinear(dim, h_dim, bias=False)
    self.down_proj = BitLinear(h_dim, dim, bias=False)
  def forward(self, x):
    return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

Tests for the BitFFN class

In [85]:
ffn_layer = BitFFN(dim=64, h_dim=256)
x_ffn = torch.randn(1, 10, 64)
output_ffn = ffn_layer(x_ffn)
assert isinstance(output_ffn, torch.Tensor), "Assertion Failed: BitFFN output should be a torch.Tensor"
assert output_ffn.shape == (1, 10, 64), f"Assertion Failed: BitFFN output shape mismatch. Expected (1, 10, 64), got {output_ffn.shape}"
assert not torch.isnan(output_ffn).any(), "Assertion Failed: BitFFN output contains NaN values"
assert not torch.isinf(output_ffn).any(), "Assertion Failed: BitFFN output contains Inf values"
print(f"BitFFN Output tensor shape: {output_ffn.shape}")

BitFFN Output tensor shape: torch.Size([1, 10, 64])
